**13/08/2026** -- Inline descriptive statistics for results section *Contributions of local and transported fire pollution*

Using Hu et al. fire PM2.5 data.

In [ ]:
library(dplyr)
library(readr)
library(dtplyr)
library(ggstats)

In [3]:
SCRATCH_DIR <- Sys.getenv("SCRATCH_DIR")
DATA_PATH   <- file.path(SCRATCH_DIR, 
                         "data/spatial/fire_pm_dep_paper_data",
                         "proc_data/df_af_annual_loc_tp_2000_2023.csv")

In [5]:
df <- read_csv(DATA_PATH)

Rows: 994412 Columns: 107
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (11): geometry, ISO_A3, country, region, continent, INCOME_GRP, ECONOMY,...
dbl (96): lon, lat, total_PM25, total_O3, fire_PM25, fire_O3, fire_PM25_hu, ...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


##### “Transported pollution contributed X% (population-weighted IQR: Y-Z%) of fire PM2.5 exposure on average, with the remainder from local fires within 100km”

2000-2017:

In [ ]:
# Compute mean across all years 2000-2023 (group by grid_id)
df_avg_17 <- lazy_dt(df) %>% 
    filter(year <= 2017) |> 
    select(c("attr_fire_PM25_emis_0_100km", "attr_fire_PM25_emis_100_2000km",
             "bc_attr_fire_PM25_emis_0_100km", "bc_attr_fire_PM25_emis_100_2000km",
             "frac_fire_PM25_emis_0_100km", "frac_fire_PM25_emis_100_2000km",
             "lon", "lat", "geometry", "country", "ISO_A3", "region", 
             "pop_count", "fire_PM25_hu", "fire_PM25")) %>% 
    group_by(lon, lat) %>% 
    summarise(
        across( c("geometry", "country", "ISO_A3", "region"), ~ first(.x) ),
        across( c("pop_count", "fire_PM25", "fire_PM25_hu"), ~ mean(.x, na.rm = TRUE) ),
        across( c('attr_fire_PM25_emis_0_100km', 'attr_fire_PM25_emis_100_2000km'), ~ mean(.x, na.rm = TRUE) ),
        across( c('bc_attr_fire_PM25_emis_0_100km', 'bc_attr_fire_PM25_emis_100_2000km'), ~ mean(.x, na.rm = TRUE) ),
        across( c('frac_fire_PM25_emis_0_100km', 'frac_fire_PM25_emis_100_2000km'), ~ mean(.x, na.rm = TRUE) ),
        .groups = "drop"
    ) %>% 
    as_tibble()

In [59]:
df_avg_17_dropna <- df_avg_17[!is.na(df_avg_17$bc_attr_fire_PM25_emis_0_100km) & 
                            !is.na(df_avg_17$bc_attr_fire_PM25_emis_100_2000km) & #nolint
                            !is.na(df_avg_17$pop_count),]

In [48]:
# Population-weighted mean of transp. fraction of fire PM, as %
100 * stats::weighted.mean(x = df_avg_17_dropna$frac_fire_PM25_emis_100_2000km,
                           w = df_avg_17_dropna$pop_count)

[1] 89.88516

In [49]:
# Population-weighted quantiles of transp. fraction of fire PM, as %
ggstats::weighted.quantile(x = df_avg_17_dropna$frac_fire_PM25_emis_100_2000km,
                           w = df_avg_17_dropna$pop_count,
                           probs = c(0.25, 0.5, 0.75)) * 100

25%      50%      75% 
86.46236 95.09234 99.06102

^^Note that it's a choice to do a population-weighted average of the *fractions* -- for the average person, what portion of their PM is transported. (Versus doing population-weighted average transported PM divided by pw-avg total PM)

2000-2022:

In [ ]:
# Compute mean across all years 2000-2023 (group by grid_id)
df_avg_22 <- lazy_dt(df) %>% 
    filter(year <= 2022) |>
    select(c("attr_fire_PM25_emis_0_100km", "attr_fire_PM25_emis_100_2000km",
             "bc_attr_fire_PM25_emis_0_100km", "bc_attr_fire_PM25_emis_100_2000km",
             "frac_fire_PM25_emis_0_100km", "frac_fire_PM25_emis_100_2000km",
             "lon", "lat", "geometry", "country", "ISO_A3", "region", 
             "pop_count", "fire_PM25_hu", "fire_PM25")) %>% 
    group_by(lon, lat) %>% 
    summarise(
        across( c("geometry", "country", "ISO_A3", "region"), ~ first(.x) ),
        across( c("pop_count", "fire_PM25", "fire_PM25_hu"), ~ mean(.x, na.rm = TRUE) ),
        across( c('attr_fire_PM25_emis_0_100km', 'attr_fire_PM25_emis_100_2000km'), ~ mean(.x, na.rm = TRUE) ),
        across( c('bc_attr_fire_PM25_emis_0_100km', 'bc_attr_fire_PM25_emis_100_2000km'), ~ mean(.x, na.rm = TRUE) ),
        across( c('frac_fire_PM25_emis_0_100km', 'frac_fire_PM25_emis_100_2000km'), ~ mean(.x, na.rm = TRUE) ),
        .groups = "drop"
    ) %>% 
    as_tibble()

In [51]:
df_avg_22_dropna <- df_avg_22[!is.na(df_avg_22$bc_attr_fire_PM25_emis_0_100km) & 
                            !is.na(df_avg_22$bc_attr_fire_PM25_emis_100_2000km) & #nolint
                            !is.na(df_avg_22$pop_count),]

In [52]:
# Population-weighted mean of transp. fraction of fire PM, as %
100 * stats::weighted.mean(x = df_avg_22_dropna$frac_fire_PM25_emis_100_2000km,
                           w = df_avg_22_dropna$pop_count)

[1] 90.12471

In [53]:
# Population-weighted quantiles of transp. fraction of fire PM, as %
ggstats::weighted.quantile(x = df_avg_22_dropna$frac_fire_PM25_emis_100_2000km,
                           w = df_avg_22_dropna$pop_count,
                           probs = c(0.25, 0.5, 0.75)) * 100

25%      50%      75% 
86.58752 95.25747 98.98557

##### “Population-weighted mean local fire pollution was X µg/m3 (range Y to Z µg/m3), compared to X µg/m3 (Y to Z µg/m3) from transported pollution”

2000-2017:

In [65]:
# Population-weighted mean of absolute local component of fire PM
stats::weighted.mean(x = df_avg_17_dropna$bc_attr_fire_PM25_emis_0_100km,
                           w = df_avg_17_dropna$pop_count)

[1] 0.3657443

In [69]:
# Range 
summary(df_avg_17_dropna$bc_attr_fire_PM25_emis_0_100km)

    Min.  1st Qu.   Median     Mean  3rd Qu.     Max. 
0.000000 0.000047 0.026176 0.380257 0.538734 4.135797 

In [64]:
# Population-weighted mean of absolute transp. component of fire PM
stats::weighted.mean(x = df_avg_17_dropna$bc_attr_fire_PM25_emis_100_2000km,
                           w = df_avg_17_dropna$pop_count)

[1] 3.369532

In [72]:
summary(df_avg_17_dropna$bc_attr_fire_PM25_emis_100_2000km)

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
 0.1845  0.8909  1.3849  3.1867  4.4056 20.5668 

2000-2022:

In [66]:
# Population-weighted mean of absolute local component of fire PM
stats::weighted.mean(x = df_avg_22_dropna$bc_attr_fire_PM25_emis_0_100km,
                           w = df_avg_22_dropna$pop_count)

[1] 0.3456596

In [73]:
summary(df_avg_22_dropna$bc_attr_fire_PM25_emis_0_100km)

    Min.  1st Qu.   Median     Mean  3rd Qu.     Max. 
0.000000 0.000119 0.024805 0.367775 0.505914 3.949421 

In [67]:
# Population-weighted mean of absolute transp. component of fire PM
stats::weighted.mean(x = df_avg_22_dropna$bc_attr_fire_PM25_emis_100_2000km,
                           w = df_avg_22_dropna$pop_count)

[1] 3.203564

In [74]:
summary(df_avg_22_dropna$bc_attr_fire_PM25_emis_100_2000km)

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
 0.1702  0.8330  1.3102  3.0630  4.2097 19.2390 